# MongoDB using PySpark

### 1. Objective

This lab explores how **Apache PySpark** can effectively analyze data sourced from two fundamentally different database models: **SQL** (Relational) and **NoSQL** (Document, like MongoDB). The core goal is to understand how PySpark's **distributed processing** capability makes it a useful for analysis, regardless of the data's original structure.

***

### 2. Database Models

| Model | Structure | Primary Focus |
| :--- | :--- | :--- |
| **SQL (Relational)** | Organized into **rigid tables** with fixed columns, requiring **joins** to link related data. | **Consistency:** Ensures data integrity through strict rules and stable links. |
| **NoSQL (Document)** | Stores data in **flexible documents** (JSON-like files), keeping an entire record self-contained. | **Evolution:** Allows the data structure to change easily and prioritizes fast lookups. |

***

While traditional **Relational Databases (SQL)** treat data as fixed rows in rigid tables, **MongoDB (NoSQL)** uses a document model. The fundamental difference lies in how each system handles **data assembly** (reading records) and **structure management** (handling changes to fields).

| Feature | MongoDB (Document Store) | Relational (Table Store) | Advantage of NoSQL |
| :--- | :--- | :--- | :--- |
| **Data Assembly** | **Self-Contained.** The entire record (e.g., product, nutrients, brand info) is stored in **one complete document**. | **Fragmented.** The record is split across multiple tables, requiring a **JOIN** operation to stitch data pieces together for every read. | **Avoids JOIN Overhead:** Eliminates the processing time needed to assemble complex records from scattered fragments. |
| **Structure Management** | **Dynamic Schema.** You can instantly add a new field (e.g., `supplier_rating`) to one product's document without affecting the database structure. | **Fixed Schema.** Requires complex, system-wide **ALTER TABLE** changes to add a field, which modifies every row and can cause system interruption. | **Zero Downtime:** Allows applications to evolve data requirements instantly without the need for time-consuming database migrations. |

### 3. Why PySpark? 🤝

PySpark's core strength is **distributed computing**—it breaks up datasets and processes their pieces across many machines simultaneously – ideally. 

PySpark achieves this structural neutrality using the **DataFrame**, which is a standardized, tabular object for distributed data:

1.  **Standardization:** PySpark uses specialized **connectors** (like the MongoDB Spark Connector) to act as a translator. These connectors read the nested data from a NoSQL document (or the fixed data from a SQL table) and automatically map it into a standard, parallel **DataFrame**.
2.  **Parallel Processing:** Once the data is in the DataFrame, PySpark uses the exact same efficient processing logic (filtering, grouping, etc.) for all data. It doesn't need to learn the database's rules; it only needs the connector to feed it chunks of data.

Therefore, whether data is tightly structured in a SQL table or loosely structured in a NoSQL document, PySpark provides a single framework for analysis.

### 4. Setup

First, we are going to install PySpark – you know the game

In [ ]:
%pip install pyspark pymongo

The next step consists in identifying the IP address of the cluster we are currently using. We use a shell command to this purpose. Copy-pasting this command, we access the Network settings of MongoDB Atlas and add the IP address (using a time limit).

In [ ]:
%%sh
curl ipecho.net/plain

The following steps are only necessary, if you are starting a new studio. If you are "awakening" an existing studio, skip them.

In [ ]:
# Install Java 17
!sudo apt-get update
!sudo apt-get install -y openjdk-17-jdk-headless

In [6]:
# Set JAVA_HOME to Java 17
import os
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"

Next, we can instantiate the Spark session.

In [ ]:
from pyspark.sql import SparkSession

from pyspark.sql import SparkSession


spark = (
    SparkSession.builder
    .appName("PySpark MongoDB")
    .config(
        "spark.jars.packages",
        "org.mongodb.spark:mongo-spark-connector_2.12:10.6.1"
    )
    .getOrCreate()
)

In [4]:
sc = spark.sparkContext

In [5]:
#spark.sparkContext.stop() -> This is only used when we need to stop Spark for some reason

Now, we need to upload a file that contains our credentials. It is not recommended to store passwords/usernames in plain code. We are going to make use of the "Secrets" functionality. 

Return to the playground and locate in the top-right corner your profile. Click on it and select "Global Settings". Click on the "Secrets" tab and enter your MongoDB username and password as separate secrets.

Add the keys corresponding to your access from MongoDB atlas. We can then access these secrets in the code. Check here the documentation for how to do it in lightning: https://lightning.ai/docs/security/security-features/secrets

In [7]:
import os

username = os.getenv("MONGODB_USERNAME")
password = os.getenv("MONGODB_PASSWORD")

We first need to create our database. We are going to do this programmatically using the ```pymongo``` library.

In [8]:
import pymongo

In [ ]:
# Now, we will unzip our database, which is stored as a JSON.
!unzip world_food_facts.zip

In [10]:
# Set MongoDB Atlas connection parameters
# mongo_uri = f"mongodb+srv://{username}:{password}@<YOUR_CLUSTER_DETAILS>" 
database = "worldfoodfacts"
collection_name = "food-facts"

In [29]:
# We will instantiate the client
client = pymongo.MongoClient(mongo_uri)

In [23]:
# Here, we create a new database and collection
db = client[database]
collection = db[collection_name]

In [ ]:
# Let's check our databases
client.list_database_names()

We could also choose use to insert data using MongoDB's GUI (MongoDB Compass), but we will do it programmatically through the Python client.

In [33]:
# Since the free cluster has storage limits, make sure to drop the database, in case you get an error and try again.
# You might need to go to the Atlas admin panel to do that (under "Collections")

In [27]:
import json

# We open the json file containing our database and we insert it using ```insert_many()````
with open("world_food_facts.json", 'r', encoding='utf-8') as f:
    data_to_insert = json.load(f)

collection.insert_many(data_to_insert)

# Don't forget to close connection
client.close()

### 5. Loading data from MongoDB into PySpark

Having uploaded our JSON database, we can now proceed to the analysis using Spark. We created our PySpark cluster with the Spark-MongoDB connector as an external library. This allows us to read data directly from MongoDB into a PySpark DataFrame.

In [11]:
# Reading our data
# We specify the source

Let's inspect the schema. You should see a lot of nested columns – which will be great fun.

In [ ]:
# CODE HERE

We can now take a look at the first few rows of the dataframe.

In [ ]:
# CODE HERE

PySpark automatically read the document collection into a PySpark DataFrame, even though – as we have just seen before — a MongoDB database does not have a fixed schema per se.

We can pre-query our database with the ``pipeline`` option. You can learn more about this here: [Aggregation Pipelines in MongoDB](https://www.mongodb.com/docs/manual/core/aggregation-pipeline/). This allows us to push down some of the computations to the database instead of using the PySpark cluster.

This table summarizes all the dollar-sign (`$`) syntax used in the aggregation pipelines in this notebook, according to their function. MongoDB has a comprehensive documentation on its query language, which you can find here: [Query Language Manual](https://www.mongodb.com/docs/manual/reference/mql/).

| Category | Operator | Type | Purpose in the Query |
| :--- | :--- | :--- | :--- |
| **Pipeline Stage** | `$match` | Stage | **Filters** documents based on specified criteria (like a SQL `WHERE` clause). |
| | `$unwind` | Stage | **Deconstructs** an array field (e.g., `countries_tags`), creating a separate document for every element. |
| | `$group` | Stage | **Aggregates** documents by a shared key (`_id`) to perform calculations. |
| | `$sort` | Stage | **Orders** the documents based on a specified field (e.g., placing the highest averages first). |
| | `$limit` | Stage | **Restricts** the number of documents passed to the next stage (used to select the Top N results). |
| | `$project` | Stage | **Reshapes** the output by including, excluding, or renaming fields for the final result. |
| **Accumulator** | `$sum` | Expression | **Counts/Sums** numeric values within a group. Used as `$sum: 1` to count documents. |
| | `$avg` | Expression | **Calculates the mean** of a numeric field across all documents in a group. |
| **Expression/Operator** | `$ne` | Operator | **"Not Equal"** (used in `$match` to filter out values like `null` or empty arrays `[]`). |
| | `$round` | Expression | **Rounds** a numeric value to a specified number of decimal places for cleaner output. |
| | `$-1` | Constant | Denotes **descending** order when used in the `$sort` stage. |
| **Field Reference** | `$field` | Reference | Used to access the **value of a field** within the document currently being processed (e.g., `"$countries_tags"`). |

In [ ]:
# Here, we query for data only from Portugal

Of course, we can make our queries more elaborate using MongoDB query language. Below, you find some useful points about how it works. 

| Component | What it Does | Example |
| :--- | :--- | :--- |
| **Stage Operator** | The action (filter, group, sort). Always starts with a **`$`**. | `$match`, `$group` |
| **Field Reference** | Points to a field's value in the document. Also starts with a **`$`**. | `"$countries_tags"` |
| **Accumulator** | A function inside `$group` that calculates a single metric across many documents. | `$sum`, `$avg` |

We specify the various ```stages``` of the pipeline as nested Python dictionaries. 

In [71]:
sugar_query = [
    # STAGE 1: Unwind the main nutrient array to access individual nutrient objects.
    
    # STAGE 2: Filter the documents to keep only the "sugars" object and records with valid countries data.
    
    # STAGE 3: Unwind the countries array to create a unique row for every country-product combination.
    
    # STAGE 4: Group documents by country tag and calculate the average sugar content and product count.
    
    # STAGE 5: Filter out countries that appear only once.
    
    # STAGE 6: Sort the results by descending average sugar content.
    
    # STAGE 7: Limit the final output to the top 5 countries (of those remaining).
    
    # STAGE 8: Reshape the output for cleanliness and readability.
]

In [ ]:
# Querying our dataframe

Naturally, we can also translate queries using ```aggregation.pipeline``` to pure PySpark. We will first take a look at how to implement the MongoDB query and then how we can reproduce the same steps in native PySpark.

In [35]:
healthy_foods_query = [
        # STAGE 1: $match

        # STAGE 2: $unwind
        # Purpose: Deconstruct the 'categories_tags' array.

        # STAGE 3: $group
        # Purpose: Aggregate the documents to count the number of healthy products per category.

        # STAGE 4: $sort
        # Purpose: Order the aggregated results to see the top categories first.

        # STAGE 5: $project
        # Purpose: Reshape and clean up the final output structure.
]

In [36]:
# Querying the database

Let us now translate the MongoDB logic into PySpark logic. We will meet several old acquaintances from previous labs that reside in the ```functions``` module.

In [28]:
from pyspark.sql.functions import col, explode, split, element_at, count, lit, desc

# ==============================================================================
# STAGE 1: $match (PySpark: filter)
# Filter for products that are Nova Group 1 or 2 AND Nutri-Score A or B.
# ==============================================================================
filtered_df = df.filter(...)

# ==============================================================================
# STAGE 2: $unwind (PySpark: explode)
# Deconstruct the 'categories_tags' array into individual rows.
# ==============================================================================
unwound_df = filtered_df.withColumn(
    ...
)


# ==============================================================================
# STAGE 3 & 4: $group & $project (PySpark: split, element_at, groupBy, agg, select)
# Group by main category, count, and rename the fields.
# ==============================================================================

# Split the tag (e.g., 'en:snacks' -> ['en', 'snacks']) and extract the main category (index 2)
grouped_df = unwound_df.withColumn(
    ...
)

# Final $project step: Select and order the final columns.

We can now write our analysis table to a new collection in the database.

In [35]:
# Differences between the read/write operations
# 1. We now use the ``write`` attribute of our DataFrame
# 2. We use the ``overwrite`` mode
# 3. We specify the ``spark.mongodb.write.connection.uri`` instead of read.connection